In [1]:
import pandas as pd

from sklearn import datasets

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

from sklearn.model_selection import train_test_split

import mlflow

from mlflow.models import infer_signature

In [2]:
#Setting tracking URI
mlflow.set_tracking_uri(uri = "http://127.0.0.1:5000")

In [3]:
#Loading the dataset
X, y = datasets.load_iris(return_X_y = True)

In [4]:
#Splitting the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20)

In [5]:
#Defining the model hyperparameters
params = {"solver" : "lbfgs", "max_iter" : 1000, "multi_class" : "auto", "random_state" : 8888, "penalty" : "l2"}

In [6]:
#Training the model
lr = LogisticRegression(**params)

lr.fit(X_train, y_train)

c:\Users\Tough\OneDrive\Desktop\Tough\MLOPS\ML_Flow\My_Class_Workings\MlFlow_Starter\venv\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='auto', random_state=8888)

In [7]:
#Predicting the test dataset
y_pred = lr.predict(X_test)

In [8]:
#Calculating the accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy

1.0

In [9]:
#MLFlow Tracking URI
mlflow.set_tracking_uri(uri = "http://127.0.0.1:5000")  #Running this step again

In [ ]:
#MLFlow experiment
mlflow.set_experiment("MLFLOW Quickstart")

#Starting MLFlow run
with mlflow.start_run():
    
    #logging the hyperparameters
    mlflow.log_params(params)
    
    #logging the accuracy metrics
    mlflow.log_metric("accuracy", accuracy)
    
    #setting a tag that we can use to remind ourselves what this run was about
    mlflow.set_tag("Training Info", "Basic Logistic Regression Model for IRIS Dataset") 
    
    #Infering the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    #logging the model
    model_info = mlflow.sklearn.log_model(
        sk_model = lr,
        artifact_path = "iris_model",
        signature = signature,
        input_example = X_train,
        registered_model_name = "iris_model"
        #This step is not recommended for real life scenarios, because until we verify whether the model is 
        #best or not, we should not register the model  
    )

    

Registered model 'iris_model' already exists. Creating a new version of this model...
2025/02/02 09:33:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_model, version 5


🏃 View run gifted-snail-352 at: http://127.0.0.1:5000/#/experiments/931875604973068958/runs/a3441bbe64e649d0ad3c7a9a51a32438
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/931875604973068958


Created version '5' of model 'iris_model'.


In [19]:
model_info.model_uri

'runs:/a3441bbe64e649d0ad3c7a9a51a32438/iris_model'

# **Inferencing and validating the model**

### 1st Method:

In [ ]:
#Copied from the MLflow dashboard

import mlflow

model_uri = model_info.model_uri
# This is the input example logged with the model
pyfunc_model = mlflow.pyfunc.load_model(model_uri)
input_data = pyfunc_model.input_example

# Verify the model with the provided input data using the logged dependencies.
# For more details, refer to:
# https://mlflow.org/docs/latest/models.html#validate-models-before-deployment
mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="local",
    
)

2025/02/02 09:43:34 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.
2025/02/02 09:43:34 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


{"predictions": [2, 0, 2, 1, 2, 0, 0, 1, 1, 1, 1, 1, 0, 2, 2, 0, 1, 1, 1, 0, 0, 0, 2, 1, 2, 1, 2, 2, 1, 2, 2, 0, 2, 1, 0, 2, 0, 0, 0, 2, 2, 0, 0, 2, 0, 2, 0, 2, 1, 1, 1, 2, 2, 2, 1, 1, 2, 0, 2, 0, 0, 2, 1, 1, 0, 0, 0, 2, 1, 1, 0, 1, 1, 0, 0, 1, 2, 2, 2, 0, 2, 2, 0, 0, 2, 2, 2, 0, 1, 1, 0, 2, 1, 1, 0, 0, 1, 0, 1, 2, 1, 2, 0, 0, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, 0, 1, 1, 1, 0]}

### 2nd Method:

In [20]:
import pandas as pd

In [23]:
#Loading the model back for prediction as a generic python function model
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
#loads the trained model stored at the path --> "model_info.model_uri"
#"mlflow.pyfunc" allows us to treat the model as a generic Python function, meaning we can call .predict() on it
#"model_info.model_uri" contains the path where the trained model was saved in MLflow

predictions = loaded_model.predict(X_test)
#"loaded_model.predict(X_test)" uses the loaded model to predict the output for X_test

iris_features_name = datasets.load_iris().feature_names
#datasets.load_iris().feature_names returns: 
#['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
#These names are used as column headers for the test dataset

result = pd.DataFrame(X_test, columns = iris_features_name)
#Here "X_test" (which is a NumPy array) is converted into a pandas DataFrame

result["actual_class"] = y_test
#"y_test" contains the actual labels (ground truth)
#A new column "actual_class" is added to the result DataFrame

result["predicted_class"] = predictions
#"predictions" contains the predicted class labels
#A new column "predicted_class" is added to the result DataFrame

print(result.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                6.2               2.9                4.3               1.3   
1                6.4               2.9                4.3               1.3   
2                5.6               3.0                4.5               1.5   
3                6.9               3.1                5.4               2.1   
4                5.7               4.4                1.5               0.4   

   actual_class  predicted_class  
0             1                1  
1             1                1  
2             1                1  
3             2                2  
4             0                0  


### **Model Registry:**

**The MLFLOW Model Registry component is a centralized model store, set of APIs, and UI, to collaboratively manage the full life cycle of an MLFLOW Model. It provides model lineage (which MLFLOW experiment and run produced the model), model versioning, model aliasing, model tagging, and annotations.**

#### **Key Features of MLflow Model Registry:**

✅ Model Versioning → Keeps track of multiple versions of a model.

✅ Model Staging → Supports lifecycle management with stages: None, Staging, Production, Archived.

✅ Model Lineage & Metadata → Stores model metadata, including the author, date, training environment, and parameters.

✅ Access Control & Annotations → Users can add descriptions, comments, and control access.

✅ Approval Workflow → Models can be reviewed and transitioned to different stages before deployment.

In [24]:
#MLFlow experiment
mlflow.set_experiment("MLFLOW Quickstart")

#Starting MLFlow run
with mlflow.start_run():
    
    #logging the hyperparameters
    mlflow.log_params(params)
    
    #logging the accuracy metrics
    mlflow.log_metric("accuracy", 1.0)
    
    #setting a tag that we can use to remind ourselves what this run was about
    mlflow.set_tag("Training Info2", "Basic Logistic Regression Model for IRIS Dataset") 
    
    #Infering the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    #logging the model
    model_info = mlflow.sklearn.log_model(
        sk_model = lr,
        artifact_path = "iris_model",
        signature = signature,
        input_example = X_train
        #registered_model_name = "iris_model"
        #This step is not recommended for real life scenarios, because until we verify whether the model is 
        #best or not, we should not register the model  
    )

    

🏃 View run whimsical-moth-382 at: http://127.0.0.1:5000/#/experiments/931875604973068958/runs/7ea4480e37a146f094b16c6c128bef6d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/931875604973068958


In [25]:
#Now how to predict using the best model? Let's find out

#Inferencing model from the model registry 
import mlflow.sklearn
model_name = "iris_model"
model_version = "latest"

model_uri = f"models:/{model_name}/{model_version}"

model = mlflow.sklearn.load_model(model_uri)

model

c:\Users\Tough\OneDrive\Desktop\Tough\MLOPS\ML_Flow\My_Class_Workings\MlFlow_Starter\venv\lib\site-packages\mlflow\store\artifact\utils\models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


LogisticRegression(max_iter=1000, multi_class='auto', random_state=8888)

In [26]:
model_uri

'models:/iris_model/latest'

In [27]:
y_pred_new = model.predict(X_test)

print(y_pred_new)

[1 1 1 2 0 0 0 2 1 0 0 0 2 2 0 0 2 1 0 2 1 1 0 0 0 1 2 1 2 2]
